In [3]:
import numpy as np

def prepare_binary_masks(masks):
    # masks shape: (N, 256, 256, 6)
    # sum all cell type channels (0-4), ignore background channel (5)
    binary_mask = masks[..., :5].sum(axis=-1)
    binary_mask = (binary_mask > 0).astype(np.float32)
    return binary_mask  # shape: (N, 256, 256)

In [5]:
!pip install segmentation-models-pytorch -q

In [6]:
!pip install segmentation-models-pytorch albumentations -q

In [7]:
!pip install -q datasets huggingface_hub

In [8]:
from datasets import load_dataset
from datasets import concatenate_datasets

dataset = load_dataset("RationAI/PanNuke")

README.md: 0.00B [00:00, ?B/s]

data/fold1-00000-of-00001.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

data/fold2-00000-of-00001.parquet:   0%|          | 0.00/264M [00:00<?, ?B/s]

data/fold3-00000-of-00001.parquet:   0%|          | 0.00/289M [00:00<?, ?B/s]

Generating fold1 split:   0%|          | 0/2656 [00:00<?, ? examples/s]

Generating fold2 split:   0%|          | 0/2523 [00:00<?, ? examples/s]

Generating fold3 split:   0%|          | 0/2722 [00:00<?, ? examples/s]

In [9]:
# Inspect overall dataset structure
print("Dataset object:")
print(dataset)

print("\nAvailable folds:")
print(dataset.keys())

Dataset object:
DatasetDict({
    fold1: Dataset({
        features: ['image', 'instances', 'categories', 'tissue'],
        num_rows: 2656
    })
    fold2: Dataset({
        features: ['image', 'instances', 'categories', 'tissue'],
        num_rows: 2523
    })
    fold3: Dataset({
        features: ['image', 'instances', 'categories', 'tissue'],
        num_rows: 2722
    })
})

Available folds:
dict_keys(['fold1', 'fold2', 'fold3'])


In [10]:
fold1 = dataset["fold1"]
fold2 = dataset["fold2"]
fold3 = dataset["fold3"]

In [11]:
data = concatenate_datasets([dataset["fold1"], dataset["fold2"], dataset["fold3"]])

In [12]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="efficientnet-b4",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None  # we apply sigmoid manually
)

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

In [4]:
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
class PanNukeSMPDataset(Dataset):
    def __init__(self, hf_dataset, transforms=None):
        self.data = hf_dataset
        self.transforms = transforms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        image = np.array(sample["image"])
        mask  = np.array(sample["instances"])   # FINAL KEY

        # merge instances → binary
        mask = np.sum(mask, axis=-1)
        mask = (mask > 0).astype(np.float32)

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask)
            image = augmented["image"]
            mask  = augmented["mask"]

        mask = mask.unsqueeze(0)

        return image, mask
# Augmentations
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), 
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), 
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [14]:
print(dataset['fold1'].column_names)

['image', 'instances', 'categories', 'tissue']


In [13]:
import torch.nn as nn

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode='binary')
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, pred, target):
        return (self.bce_weight * self.bce(pred, target) + 
                self.dice_weight * self.dice(pred, target))

In [ ]:
fold_scores = []

folds = [dataset["fold1"], dataset["fold2"], dataset["fold3"]]

for i in range(3):

    print(f"\n===== FOLD {i+1} =====")

    val_data = folds[i]
    train_data = concatenate_datasets([folds[j] for j in range(3) if j != i])

    train_dataset = PanNukeSMPDataset(train_data, transforms=train_tfms)
    val_dataset   = PanNukeSMPDataset(val_data, transforms=get_val_transforms())

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False)

    model = smp.Unet(
        encoder_name="efficientnet-b4",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
        activation=None
    )

    model = model.to(device)

    val_dice = train_model(model, train_loader, val_loader, fold=i+1)

    fold_scores.append(val_dice)

    torch.cuda.empty_cache()

print("\nFinal Results:")
print("Fold Scores:", fold_scores)
print("Mean Dice:", sum(fold_scores)/len(fold_scores))

In [ ]:
def train_model(model, train_loader, val_loader, fold, epochs=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=2, factor=0.5
    )

    best_dice = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for images, masks in train_loader:
            images = images.to(device)
            masks  = masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            loss = combined_loss(outputs, masks)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # validation
        model.eval()
        val_dice = 0

        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device)
                masks  = masks.to(device)

                outputs = model(images)
                val_dice += dice_score(outputs, masks)

        val_dice /= len(val_loader)

        scheduler.step(val_dice)

        print(f"[Fold {fold}] Epoch {epoch+1}: Loss={train_loss:.4f}, Dice={val_dice:.4f}")

        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), f"best_fold_{fold}.pth")

    return best_dice

In [15]:
def prepare_instance_mask(instances):

    instances = np.array(instances)

    # create empty instance map
    h, w = 256, 256
    instance_map = np.zeros((h, w), dtype=np.int32)

    for i, inst in enumerate(instances):
        instance_map[inst > 0] = i + 1

    return instance_map

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# Training Params
EPOCHS = 20
BATCH_SIZE = 16
LR = 1e-4
SUBSET = None

# -------------------------------
# Dataset preparation functions
# -------------------------------

def prepare_image(img):

    img = np.array(img)

    if len(img.shape) == 2:
        img = np.stack([img]*3, axis=-1)

    if img.shape[-1] == 4:
        img = img[..., :3]

    return img


def prepare_instance_mask(instances):

    instances = np.array(instances)

    instance_map = np.zeros((256,256), dtype=np.int32)

    for i, inst in enumerate(instances):
        instance_map[inst > 0] = i + 1

    return instance_map


# -------------------------------
# Transforms
# -------------------------------

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Normalize(),
    ToTensorV2()
])


# -------------------------------
# Dataset Class
# -------------------------------

class PanNukeDataset(Dataset):
    
    def __init__(self, images, masks, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        
        img = self.images[idx]
        mask = self.masks[idx]
        
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img = aug["image"]
            mask = aug["mask"]
            
        return img, mask.unsqueeze(0).float()


# -------------------------------
# Load HuggingFace dataset
# -------------------------------

from datasets import load_dataset

dataset = load_dataset("RationAI/PanNuke")

fold_images = []
fold_masks = []

for i in range(1,4):

    fold = dataset[f'fold{i}']

    imgs = []
    msks = []

    for img, inst in zip(fold['image'], fold['instances']):
        
        img = prepare_image(img)
        inst_mask = prepare_instance_mask(inst)

        imgs.append(img)
        msks.append(inst_mask)

    fold_images.append(np.stack(imgs))
    fold_masks.append(np.stack(msks))


# -------------------------------
# Training Loop
# -------------------------------

for fold in range(3):

    print(f"\nTraining Fold {fold+1}")

    val_imgs = fold_images[fold]
    val_msks = fold_masks[fold]

    train_imgs = np.concatenate(
        [fold_images[i] for i in range(3) if i != fold]
    )

    train_msks = np.concatenate(
        [fold_masks[i] for i in range(3) if i != fold]
    )

    # Subset for faster training
    train_imgs = train_imgs[:SUBSET]
    train_msks = train_msks[:SUBSET]

    train_ds = PanNukeDataset(train_imgs, train_msks, train_transform)
    val_ds = PanNukeDataset(val_imgs, val_msks, val_transform)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True
    )

    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE
    )


    # -------------------------------
    # Model (EfficientNet B0)
    # -------------------------------

    model = smp.Unet(
        encoder_name="efficientnet-b0",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
        activation=None
    ).to(DEVICE)


    # Loss + Optimizer

    loss_fn = smp.losses.DiceLoss(mode="binary")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR
    )


    # -------------------------------
    # Training
    # -------------------------------

    for epoch in range(EPOCHS):

        model.train()
        train_loss = 0

        loop = tqdm(train_loader)

        for imgs, masks in loop:

            imgs = imgs.to(DEVICE)
            masks = masks.to(DEVICE)

            preds = model(imgs)

            loss = loss_fn(preds, masks)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            loop.set_description(f"Epoch {epoch+1}")
            loop.set_postfix(loss=loss.item())


        print(f"Epoch {epoch+1} Loss: {train_loss/len(train_loader)}")

Using device: cuda

Training Fold 1


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Epoch 1: 100%|██████████| 328/328 [01:12<00:00,  4.53it/s, loss=0.897]


Epoch 1 Loss: 0.9003771330888678


Epoch 2: 100%|██████████| 328/328 [01:14<00:00,  4.42it/s, loss=0.865]


Epoch 2 Loss: 0.8896899719427271


Epoch 3:  41%|████      | 133/328 [00:31<00:44,  4.35it/s, loss=0.883]

In [ ]:
import matplotlib.pyplot as plt

model.eval()

imgs, masks = next(iter(val_loader))

imgs = imgs.to(DEVICE)

with torch.no_grad():
    preds = model(imgs)

preds = torch.sigmoid(preds)
preds = (preds > 0.5).float()

imgs = imgs.cpu()
masks = masks.cpu()
preds = preds.cpu()


for i in range(3):

    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.title("Image")
    plt.imshow(imgs[i].permute(1,2,0))

    plt.subplot(1,3,2)
    plt.title("Ground Truth")
    plt.imshow(masks[i][0])

    plt.subplot(1,3,3)
    plt.title("Prediction")
    plt.imshow(preds[i][0])

    plt.show()

In [ ]:
def dice_score(pred, target, threshold=0.5):
    pred = torch.sigmoid(pred)
    pred = (pred > threshold).float()

    intersection = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))

    dice = (2 * intersection + 1e-8) / (union + 1e-8)
    return dice.mean().item()

In [ ]:
dice = dice_score(preds, masks)
print("Dice Score:", dice.item())